# Systematic Alpha Lab: Consolidated Platform Demo

This notebook checks that the consolidated package is importable, loads the platform configs, and shows the implemented layers now available from the parent repo.

## 1. Project Setup

If the package was installed with `python -m pip install -e ".[dev]"`, the imports below should work directly. The path fallback keeps the notebook usable if the Jupyter kernel is not using the same environment.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/Users/edl/Documents/dev/quantlab_v2/quantlab_step0_intro")
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

## 2. Import Consolidated Layers

## 3. Run the Simple End-to-End Demo

This synthetic workflow is the easiest way to understand the platform. It does not need WRDS, Alpha Vantage, or local parquet files.

In [ ]:
from systematic_alpha_lab.workflows import run_synthetic_equity_alpha_demo

data, factor_result, alpha_result = run_synthetic_equity_alpha_demo()

print(f"Synthetic prices: {data.prices.shape[0]} dates x {data.prices.shape[1]} assets")
print(f"Sectors: {data.sectors.nunique()}")
print("Factor names:", list(factor_result.factors))
print("Alpha metrics:", {k: v for k, v in alpha_result.metrics.items() if k != "decay"})

In [ ]:
import pandas as pd

pd.DataFrame(factor_result.analytics).T

In [ ]:
import systematic_alpha_lab

from systematic_alpha_lab.data_pipeline import get_final_data, transform_raw_to_final, run_quality_checks
from systematic_alpha_lab.factor_research import DataLoader, FactorBase
from systematic_alpha_lab.factor_research.factor_definitions import get_default_factors
from systematic_alpha_lab.alpha import AlphaDataLoader, evaluate_alpha, run_alpha_pipeline

print("Package:", systematic_alpha_lab.__name__)
print("Data layer:", get_final_data.__name__, transform_raw_to_final.__name__, run_quality_checks.__name__)
print("Factor layer:", DataLoader.__name__, FactorBase.__name__)
print("Alpha layer:", AlphaDataLoader.__name__, evaluate_alpha.__name__, run_alpha_pipeline.__name__)

## 4. Inspect Configs

These configs are the platform-level controls for data defaults, factor cleaning/composites, and alpha construction.

In [ ]:
import json
import yaml

data_config = yaml.safe_load((PROJECT_ROOT / "config/datalist.yml").read_text())
factor_config = json.loads((PROJECT_ROOT / "config/factors/config.json").read_text())
alpha_config = json.loads((PROJECT_ROOT / "config/alpha/config.json").read_text())

print("Data config sections:", list(data_config.keys()))
print("Factor config sections:", list(factor_config.keys()))
print("Alpha config sections:", list(alpha_config.keys()))

In [ ]:
implemented_composites = list(factor_config.get("composites", {}).keys())
alpha_pods = list(alpha_config.get("pods", {}).keys())

print("Composite themes:")
for name in implemented_composites:
    print("-", name)

print("\nAlpha pods:")
for name in alpha_pods:
    print("-", name)

## 5. Factor Catalog

The consolidated factor layer exposes the existing factor catalog through `get_default_factors()`.

In [ ]:
default_factors = get_default_factors()

print(f"Number of default factors: {len(default_factors)}")
print("First 15 factors:")
for factor in default_factors[:15]:
    print("-", getattr(factor, "name", factor.__class__.__name__))

## 6. Data Artifact Check

The full pipeline expects local parquet artifacts under `../data`. This cell only checks what is available locally; it does not call external APIs.

In [ ]:
data_root = PROJECT_ROOT.parent / "data"
processed_dir = data_root / "data-processed"
factors_dir = data_root / "factors"
alpha_dir = data_root / "alpha"

for label, path in {
    "data_root": data_root,
    "processed_dir": processed_dir,
    "factors_dir": factors_dir,
    "alpha_dir": alpha_dir,
}.items():
    print(f"{label}: {path} | exists={path.exists()}")

In [ ]:
for path in [processed_dir, factors_dir, alpha_dir]:
    if path.exists():
        files = sorted(path.glob("*.parquet"))[:10]
        print(f"\n{path.name}: showing {len(files)} sample parquet files")
        for file in files:
            print("-", file.name)

## 7. Next Development Layer

The correct next platform layer is the risk model. Portfolio construction should consume risk-model outputs rather than directly turning alpha ranks into weights.

In [ ]:
planned_layers = {
    "risk": "factor exposures, covariance, beta, idiosyncratic risk, risk contribution",
    "portfolio": "constraints, optimizers, target weights, turnover/liquidity controls",
    "backtest": "rebalance simulation, transaction costs, slippage, attribution",
    "agents": "hypothesis, experiment, evaluation, report generation",
}

for layer, purpose in planned_layers.items():
    print(f"{layer}: {purpose}")